In [ ]:
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes

print("Python:", __import__("sys").version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nLibraries:")
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/SupportOpsAI"

print(os.listdir(PROJECT_DIR))

In [ ]:
import pandas as pd

processed_path = f"{PROJECT_DIR}/data/processed"

train_df = pd.read_csv(f"{processed_path}/train.csv")
val_df = pd.read_csv(f"{processed_path}/validation.csv")
test_df = pd.read_csv(f"{processed_path}/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded")
print("Pad token:", tokenizer.pad_token)

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """You are a customer support ticket classifier.

Choose exactly one queue from:
Technical Support
Product Support
Customer Service
IT Support
Billing and Payments
Returns and Exchanges
Service Outages and Maintenance
Sales and Pre-Sales
Human Resources
General Inquiry

Choose exactly one priority from:
low
medium
high

Return ONLY valid JSON in exactly this format:
{"queue": "<queue>", "priority": "<priority>"}
"""


def format_ticket(row):
    subject = row["subject"] if pd.notna(row["subject"]) else ""

    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": (
                    f"Subject: {subject}\n\n"
                    f"Body: {row['body']}"
                )
            },
            {
                "role": "assistant",
                "content": (
                    f'{{"queue": "{row["queue"]}", '
                    f'"priority": "{row["priority"]}"}}'
                )
            }
        ]
    }


train_dataset = Dataset.from_list(
    train_df.apply(format_ticket, axis=1).tolist()
)

val_dataset = Dataset.from_list(
    val_df.apply(format_ticket, axis=1).tolist()
)

test_dataset = Dataset.from_list(
    test_df.apply(format_ticket, axis=1).tolist()
)

print(train_dataset)
print(val_dataset)
print(test_dataset)

In [ ]:
def apply_chat_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return example


train_dataset = train_dataset.map(apply_chat_template)
val_dataset = val_dataset.map(apply_chat_template)
test_dataset = test_dataset.map(apply_chat_template)

In [ ]:
print(train_dataset[0]["text"])

In [ ]:
import numpy as np

lengths = []

for example in train_dataset:
    tokens = tokenizer(
        example["text"],
        add_special_tokens=False
    )["input_ids"]

    lengths.append(len(tokens))

print("Min:", min(lengths))
print("Max:", max(lengths))
print("Average:", np.mean(lengths))